# 01 — PyMarketSim Walkthrough

End-to-end familiarisation with the PyMarketSim codebase. Two parts:

1. **Low-level API** — build a `Market`, populate it with `ZIAgent`s, and step the simulator manually.
2. **Gym-wrapper API** — use the `MMEnv` / `SPEnv` / `MMSPEnv` Gym interfaces that the RL training loop will consume.

Goal: confirm every API surface we will rely on, document it in plain English, and surface any conflicts with `docs/methodology.md`.

## 0 — Imports & Setup

In [ ]:
import math
import random

import numpy as np
import torch
from marketsim.agent.noise_ZI_agent import ZIAgent
from marketsim.fourheap.constants import BUY, SELL
from marketsim.fundamental.lazy_mean_reverting import LazyGaussianMeanReverting
from marketsim.market.market import Market
from marketsim.wrappers.metrics import (
    midprice_move,
    queue_imbalance,
    realized_volatility,
    relative_strength_index,
    signed_volume,
    volume_imbalance,
)

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

## Part 1 — Low-Level API

The simulator is built around three primitives:

| Primitive | Class | What it does |
|---|---|---|
| **Fundamental** | `GaussianMeanReverting` or `LazyGaussianMeanReverting` | Generates a mean-reverting price series |
| **Market** | `Market` | Holds a `FourHeap` order book + event queue; calls `step()` to process arrivals and match orders |
| **Agent** | `ZIAgent`, `MMAgent`, `SpoofingAgent`, `HBLAgent`, … | Each has `take_action() -> List[Order]` and `update_position(q, cash)` |

### 1.1 — Fundamental value process

Two variants:
- `GaussianMeanReverting(mean, final_time, r, shock_var)` — **eager**, pre-generates all values at construction time.
- `LazyGaussianMeanReverting(mean, final_time, r, shock_var)` — **lazy**, generates on demand via `get_value_at(t)`. This is what the wrappers use.

Both expose `get_info() -> (mean, r, final_time)` and `get_final_fundamental()`.

In [ ]:
SIM_TIME = 200
MEAN = 1e5
R = 0.05
SHOCK_VAR = 5e6

fund = LazyGaussianMeanReverting(mean=MEAN, final_time=SIM_TIME + 1, r=R, shock_var=SHOCK_VAR)

print(f"Fundamental info (mean, r, T): {fund.get_info()}")
print(f"Fundamental at t=0:   {fund.get_value_at(0):.2f}")
print(f"Fundamental at t=50:  {fund.get_value_at(50):.2f}")
print(f"Fundamental at t=200: {fund.get_value_at(200):.2f}")
print(f"Final fundamental:    {fund.get_final_fundamental():.2f}")

### 1.2 — Market & Order Book

`Market(fundamental, time_steps)` creates:
- `market.order_book` — a `FourHeap` (4 heaps: buy_matched, buy_unmatched, sell_matched, sell_unmatched)
- `market.matched_orders` — list of `MatchedOrder` from all previous clearings
- `market.event_queue` — queues `Order` objects by time

Key methods:
- `add_orders(orders)` — schedules orders into the event queue
- `step()` — processes all orders at current time, inserts into book, clears market, returns list of `MatchedOrder`
- `withdraw_all(agent_id)` — removes all orders by an agent from the book
- `get_fundamental_value()` / `get_final_fundamental()` / `get_midprices()`
- `reset(fundamental)` — reinitialises book, event queue, matched_orders

In [ ]:
market = Market(fundamental=fund, time_steps=SIM_TIME)

print(f"Order book type: {type(market.order_book).__name__}")
print(f"Best bid (empty book): {market.order_book.get_best_bid()}")  # -inf when empty
print(f"Best ask (empty book): {market.order_book.get_best_ask()}")  # +inf when empty
print(f"Midprices so far: {market.get_midprices()}")

### 1.3 — Agents

All agents inherit from `Agent` ABC. Key interface:

```python
agent.get_id() -> int
agent.take_action(side) -> List[Order]   # side=BUY or SELL for most agents
agent.update_position(quantity, cash)
agent.reset()
agent.position  # current inventory (int)
agent.cash      # current cash (float)
```

**ZIAgent** — Zero-intelligence. Estimates fundamental via `(1-rho)*mean + rho*val`, adds shade offset + private value, returns a single `Order`.

**MMAgent (simple)** — Places K bid + K ask orders around `estimate +/- omega/2`, spaced by `xi`.

**MMAgentBeta** — Uses Beta distribution to allocate volume across levels. `take_action(action)` accepts 4 params (a_buy, b_buy, a_sell, b_sell) for RL control.

**SpoofingAgent** — `take_action(action)` where action = (regular_order_price, spoofing_order_price) both normalised [0,1]. Places a real SELL order + a spoof BUY order.

**HBLAgent** — Heuristic Belief Learning. Uses `fastcubicspline` to compute belief functions over order flow history. Falls back to ZI when insufficient trades.

In [ ]:
Q_MAX = 10
PV_VAR = 5e6
SHADE = [10, 30]

agents = {}
for i in range(5):
    agents[i] = ZIAgent(
        agent_id=i, market=market, q_max=Q_MAX, shade=SHADE, pv_var=PV_VAR, est_var=1e6
    )

print(f"Created {len(agents)} ZI agents")
print(f"Agent 0 position: {agents[0].position}, cash: {agents[0].cash}")

orders = agents[0].take_action(BUY)
print(f"Agent 0 take_action(BUY) -> {len(orders)} order(s)")
for o in orders:
    print(
        f"  Order: price={o.price:.2f}, side={'BUY' if o.order_type==BUY else 'SELL'}, qty={o.quantity}, agent={o.agent_id}, time={o.time}"
    )

### 1.4 — Manual Simulation Loop

The wrappers automate this, but understanding the raw loop is essential.

Each timestep:
1. Withdraw all existing orders for each arriving agent (simulates order replacement)
2. Call `agent.take_action(side)` to get new orders
3. `market.add_orders(orders)` to schedule them
4. `market.step()` to process the event queue, match orders, and get `MatchedOrder` list
5. For each `MatchedOrder`, call `agent.update_position(quantity, cash)` on the relevant agent

Arrivals are governed by a Geometric distribution: `sample_arrivals(lam, n)` gives inter-arrival times.

In [ ]:
from collections import defaultdict

import torch.distributions as dist


def sample_arrivals(p, num_samples):
    geometric_dist = dist.Geometric(torch.tensor([p]))
    return geometric_dist.sample((num_samples,)).squeeze()


LAM = 0.1  # arrival rate
N_AGENTS = 5

arrival_times = sample_arrivals(LAM, 10000)
arrivals = defaultdict(list)
for aid, idx in enumerate(range(N_AGENTS)):
    arrivals[arrival_times[idx].item()].append(aid)

print(f"Sampled arrival times (first 5 agents): {dict(list(arrivals.items())[:5])}")

In [ ]:
all_midprices = []
all_spreads = []
all_matched = []

for t in range(SIM_TIME):
    market.event_queue.set_time(t)

    arriving = arrivals.get(t, [])
    for aid in arriving:
        market.withdraw_all(aid)
        side = random.choice([BUY, SELL])
        orders = agents[aid].take_action(side)
        market.add_orders(orders)

    matched = market.step()
    for mo in matched:
        aid = mo.order.agent_id
        q = mo.order.order_type * mo.order.quantity
        cash = -mo.price * mo.order.quantity * mo.order.order_type
        agents[aid].update_position(q, cash)

    all_matched.append(len(matched))

    best_ask = market.order_book.get_best_ask()
    best_bid = market.order_book.get_best_bid()
    if not math.isinf(best_ask) and not math.isinf(best_bid):
        all_midprices.append((best_ask + best_bid) / 2)
        all_spreads.append(best_ask - best_bid)

print(f"Ran {SIM_TIME} timesteps")
print(f"Total matched orders: {sum(all_matched)}")
print(f"Midprices collected: {len(all_midprices)}")
if all_midprices:
    print(f"First 5 midprices: {[f'{m:.2f}' for m in all_midprices[:5]]}")
    print(f"Last 5 midprices:  {[f'{m:.2f}' for m in all_midprices[-5:]]}")
    print(f"Spreads (first 5):  {[f'{s:.2f}' for s in all_spreads[:5]]}")

In [ ]:
final_fund = market.get_final_fundamental()
print(f"\nFinal fundamental: {final_fund:.2f}")
print("\nAgent positions at end:")
for aid, agent in agents.items():
    val = agent.get_pos_value() + agent.position * final_fund + agent.cash
    print(f"  Agent {aid}: position={agent.position}, cash={agent.cash:.2f}, total_value={val:.2f}")

### 1.5 — Order Book Internals

`FourHeap` structure:
- `buy_unmatched` — max-heap (highest price is best, peeked with `.peek()`)
- `sell_unmatched` — min-heap (lowest price is best, peeked with `.peek()`)
- `buy_matched` / `sell_matched` — heaps for already-matched orders

Key methods:
- `insert(order)` — adds order and immediately tries to match
- `get_best_bid()` / `get_best_ask()` — returns -inf / +inf when empty
- `withdraw_all(agent_id)` — removes all orders for an agent
- `market_clear(time)` — settles matched orders, returns `MatchedOrder` list
- `observe()` — returns dict of current book state
- `update_midprice()` / `midprices` — tracks midprice history

In [ ]:
ob = market.order_book
print(f"Best bid: {ob.get_best_bid():.2f}")
print(f"Best ask: {ob.get_best_ask():.2f}")
print(f"Buy unmatched count:  {ob.buy_unmatched.count()}")
print(f"Sell unmatched count: {ob.sell_unmatched.count()}")
print(f"Buy unmatched order dict size:  {len(ob.buy_unmatched.order_dict)}")
print(f"Sell unmatched order dict size: {len(ob.sell_unmatched.order_dict)}")
print(f"\nMidprices (last 10): {[f'{m:.2f}' for m in market.get_midprices()[-10:]]}")

### 1.6 — Metrics (from `wrappers/metrics.py`)

These compute observation features used by the Gym environments:

| Metric | Formula | Range |
|---|---|---|
| `volume_imbalance` | (sell_vol - buy_vol) / (sell_vol + buy_vol) | [-1, 1] |
| `queue_imbalance` | (sell_orders - buy_orders) / (sell_orders + buy_orders) | [-1, 1] |
| `realized_volatility` | sqrt(sum(log-returns^2)) over lookback | [0, inf) |
| `relative_strength_index` | 100 - 100/(1 + RS) | [0, 100] |
| `midprice_move` | last_mid - mean(mid[-lookback:-1]) | (-inf, inf) |
| `signed_volume` | from `market.get_signed_volume()` | (-inf, inf) |

In [ ]:
print("Metrics on the current market state:")
print(f"  Volume imbalance:    {volume_imbalance(market):.4f}")
print(f"  Queue imbalance:     {queue_imbalance(market):.4f}")
print(f"  Realized volatility: {realized_volatility(market):.6f}")
print(f"  RSI:                 {relative_strength_index(market):.2f}")
print(f"  Midprice move:       {midprice_move(market):.4f}")
print(f"  Signed volume:       {signed_volume(market):.4f}")

## Part 2 — Gym Wrapper API

Three environments, all inheriting `gymnasium.Env`:

| Env | Self agent | Background | Observation dim | Action dim |
|---|---|---|---|---|
| `MMEnv` | 1 `MMAgentBeta` | N `ZIAgent` | 5 (or 10 with extras) | 2 (continuous) |
| `SPEnv` | 1 `SpoofingAgent` | N `ZIAgent` | 10 + 2*q_max | 2 (continuous) |
| `MMSPEnv` | 1 `SpoofingAgent` + 1 `MMAgent` | 12 `ZIAgent` + (N-13) `HBLAgent` | 12 | 2 (continuous) |

All use `LazyGaussianMeanReverting` and `sample_arrivals()` for geometric inter-arrival times.

### 2.1 — MMEnv (Market Maker)

**Observation** (5-dim, normalised to [0,1] or [-1,1]):
1. `time_left / sim_time`
2. `fundamental_value / normalizers["fundamental"]`
3. `best_ask / normalizers["fundamental"]` (1 if inf)
4. `best_bid / normalizers["fundamental"]` (0 if inf)
5. `MM_inventory / normalizers["invt"]`

**Action** (2-dim, continuous [-1, 1]): interpreted by `MMAgentBeta.take_action(action)` as (a_buy, b_buy, a_sell, b_sell) — wait, actually:
- The action space is `Box(-1, 1, shape=(2,))`
- But `MMAgentBeta.take_action(action)` expects 4 values if `policy=True`
- **Bug/API mismatch**: when `policy=False` (default), the MM uses static beta params and ignores the action entirely!

**Reward**: Change in MM's liquidation value (position * final_fundamental + cash) between arrivals, normalised by `normalizers["reward"]`.

**Step logic**:
1. `MM_step(action)` — withdraw MM's orders, call `MM.take_action(action)`, add to market
2. `agents_step()` — withdraw + resubmit for any background agents arriving at this time
3. `market_step()` — call `market.step()`, process matches, update positions
4. `run_until_next_MM_arrival()` — fast-forward through background-only steps until MM arrives again

In [ ]:
from marketsim.wrappers.MM_wrapper import MMEnv

mm_normalizers = {"fundamental": 1.2e5, "invt": 10, "cash": 5e5, "reward": 1e4}
beta_params = {"a_buy": 0.5, "b_buy": 0.5, "a_sell": 0.5, "b_sell": 0.5}

mm_env = MMEnv(
    num_background_agents=25,
    sim_time=500,
    lam=0.1,
    lamMM=5e-3,
    mean=1e5,
    r=0.05,
    shock_var=5e6,
    q_max=10,
    pv_var=5e6,
    shade=[250, 500],
    normalizers=mm_normalizers,
    beta_params=beta_params,
)

print(f"Observation space: {mm_env.observation_space}")
print(f"Action space:      {mm_env.action_space}")

obs, info = mm_env.reset()
print(f"\nInitial observation: {obs}")
print(f"Initial info: {info}")

In [ ]:
mm_rewards = []
mm_observations = [obs.copy()]

for step in range(50):
    action = mm_env.action_space.sample()
    obs, reward, terminated, truncated, info = mm_env.step(action)
    mm_rewards.append(reward)
    mm_observations.append(obs.copy())
    if terminated:
        print(f"Episode terminated at step {step}")
        break

print(f"Steps completed: {len(mm_rewards)}")
print(f"Cumulative reward: {sum(mm_rewards):.4f}")
print(f"Mean reward: {np.mean(mm_rewards):.4f}")
print(f"Last observation: {obs}")

In [ ]:
stats = mm_env.get_stats()
print("\nEnv stats after episode:")
print(f"  Total quantity traded: {stats['total_quantity']}")
print(f"  MM quantity traded:    {stats['MM_quantity']}")
print(f"  MM final value:        {stats['MM_value']:.2f}")
print(f"  Spreads (last 5):      {[f'{s:.2f}' for s in stats['spreads'][-5:]]}")
print(f"  Midprices (last 5):    {[f'{m:.2f}' for m in stats['midprices'][-5:]]}")
print(f"  Inventory (last 5):    {stats['inventory'][-5:]}")
print(f"  Social welfare:        {mm_env.compute_social_welfare():.2f}")

### 2.2 — SPEnv (Spoofer-only)

**Observation** (10 + 2*q_max dim): 5 base + 5 metrics + private values array.

**Action** (2-dim, continuous [0, 1]): (regular_order_price, spoofing_order_price) both normalised fractions of `normalizers["fundamental"]`.

**Reward**: Change in spoofer's liquidation value, normalised by `normalizers["fundamental"]`.

The spoofer always places a **SELL** regular order and a **BUY** spoofing order. The spoofing order is never cancelled within the step (it persists until the next arrival).

In [ ]:
from marketsim.wrappers.SP_wrapper import SPEnv

sp_normalizers = {"fundamental": 5e5, "invt": 1e3, "cash": 1e5}

sp_env = SPEnv(
    num_background_agents=25,
    sim_time=100,
    lam=0.1,
    lamSP=0.1,
    mean=1e5,
    r=0.05,
    shock_var=5e6,
    q_max=10,
    pv_var=5e6,
    shade=[250, 500],
    normalizers=sp_normalizers,
)

print(f"Observation space: {sp_env.observation_space}")
print(f"Action space:      {sp_env.action_space}")

obs, info = sp_env.reset()
print(f"\nInitial obs shape: {obs.shape}")
print(f"Initial obs: {obs}")

In [ ]:
sp_rewards = []
for step in range(50):
    action = sp_env.action_space.sample()
    obs, reward, terminated, truncated, info = sp_env.step(action)
    sp_rewards.append(reward)
    if terminated:
        print(f"Episode terminated at step {step}")
        break

print(f"Steps completed: {len(sp_rewards)}")
print(f"Cumulative reward: {sum(sp_rewards):.4f}")
print(f"Last obs shape: {obs.shape}")

### 2.3 — MMSPEnv (Market Maker + Spoofer)

The most complex environment — has a `MMAgent` (simple, non-beta) as a background agent AND a `SpoofingAgent` as the self agent.

**Background agents**: first 12 are `ZIAgent`, the rest (up to `num_agents - 2`) are `HBLAgent`. Plus one `MMAgent` (simple version from `market_maker.py`).

**Observation** (12-dim): time_left, fundamental, best_ask, best_bid, SP_inventory, midprice_delta, vol_imbalance, que_imbalance, vol, rsi, est_fundamental, midprice.

**Action** (2-dim, continuous [0.01, 1] x [0.1, 1]): (regular_order_price_frac, spoofing_order_price_frac).

**Reward**: Change in spoofer's liquidation value, normalised by `normalizers["reward"]`.

The spoofer enters at t=1000 (offset), so there's a warm-up period. The MM is a background agent that runs on its own schedule.

In [ ]:
from marketsim.wrappers.MMSP_wrapper import MMSPEnv

mmsp_normalizers = {"fundamental": 1.2e5, "invt": 1e3, "reward": 1e4}

mmsp_env = MMSPEnv(
    num_background_agents=25,
    sim_time=2000,
    lam=5e-3,
    lamSP=5e-2,
    lamMM=8e-2,
    mean=1e5,
    r=0.05,
    shock_var=5e6,
    q_max=10,
    pv_var=5e6,
    shade=[250, 500],
    xi=100,
    omega=256,
    K=8,
    normalizers=mmsp_normalizers,
    learning=False,
)

print(f"Observation space: {mmsp_env.observation_space}")
print(f"Action space:      {mmsp_env.action_space}")

obs, info = mmsp_env.reset()
print(f"\nInitial obs: {obs}")

In [ ]:
mmsp_rewards = []
for step in range(50):
    action = mmsp_env.action_space.sample()
    obs, reward, terminated, truncated, info = mmsp_env.step(action)
    mmsp_rewards.append(reward)
    if terminated:
        print(f"Episode terminated at step {step}")
        break

print(f"Steps completed: {len(mmsp_rewards)}")
print(f"Cumulative reward: {sum(mmsp_rewards):.4f}")

## Part 3 — API Conflicts with Methodology

After reading the full PyMarketSim codebase, the following conflicts or gaps with `docs/methodology.md` have been identified:

### Conflict 1: No defender / predator role abstraction
**Methodology assumes:** A defender liquidating size Q over horizon T=1800s, and K predators that detect order-flow signatures.

**PyMarketSim reality:** The existing agents are: ZI (noise), MM (market maker), SpoofingAgent (always sells real + buys spoof), HBLAgent (belief-learning). There is **no defender agent** that executes a parent order, and **no predator agent** that detects order-flow signatures from a specific counterparty.

**Impact:** We must build `DefenderAgent` and `PredatorAgent` from scratch. The `SpoofingAgent` is the closest existing agent to a predator, but its action space (place spoof + real order) doesn't match the predator's conceptual role (detect + front-run). The defender needs a parent-order execution API that doesn't exist.

### Conflict 2: Time horizon and step semantics
**Methodology assumes:** T=1800 seconds with control interval 1s (N=1800 steps).

**PyMarketSim reality:** Time is discrete integer steps. The wrappers use `sim_time` (default 1000-2000). Step semantics are event-driven (agents arrive at geometric intervals), not fixed-interval. The Gym `step()` call may advance many internal timesteps between self-agent arrivals.

**Impact:** Mapping "1800 seconds" to simulator steps requires defining a time-unit convention. The current wrappers' `run_until_next_*_arrival()` fast-forwards between RL agent arrivals, so the effective control frequency depends on arrival rate `lam`.

### Conflict 3: Observation space mismatch
**Methodology assumes:** Defender knows own inventory, time, public LOB state. Predator knows own state, time, public LOB state.

**PyMarketSim reality:** The wrappers' observations include fundamental value, best bid/ask, inventory, and 5 metrics (midprice move, volume imbalance, queue imbalance, realized volatility, RSI). But the MM wrapper **ignores the 5 extra metrics** in its actual 5-dim observation. The SP wrapper includes private values.

**Impact:** We need to define new observation spaces for Defender and Predator agents. The predator needs order-flow features (e.g., signed volume, trade arrival rate) that aren't in the current metrics.

### Conflict 4: Action space mismatch
**Methodology assumes:** Defender controls execution schedule (volume per interval). Predator controls front-running aggressiveness.

**PyMarketSim reality:** Existing action spaces are: MMEnv (2-dim beta params for order profile), SPEnv (2-dim price fractions for real+spoof order). No agent has a "volume schedule" or "front-running intensity" action space.

**Impact:** We must design action spaces that map naturally to our game. The defender's action could be (volume_to_execute, price_offset) at each arrival. The predator's could be (detection_threshold, front_run_size, price_offset).

### Conflict 5: No multi-agent Gym interface
**Methodology assumes:** Independent PPO with adversarial alternation for K+1 agents.

**PyMarketSim reality:** Each wrapper is single-agent (one self agent). The MMSPEnv has the MM as a background (non-RL) agent. There is no PettingZoo `ParallelEnv` or `AECEnv` wrapper.

**Impact:** We must build a multi-agent wrapper (likely PettingZoo ParallelEnv) that manages both defender and predator as simultaneous RL agents.

### Conflict 6: Market.reset() requires new fundamental
**PyMarketSim reality:** `Market.reset(fundamental)` requires passing a new fundamental object. The `LazyGaussianMeanReverting` has a `_generate()` method for regenerating values. This is important for training diversity.

**Not a conflict per se**, but important for our training loop design.

### Non-conflict: GaussianMeanReverting vs LazyGaussianMeanReverting
The MMSPEnv uses the eager `GaussianMeanReverting`; the MM/SP wrappers use the lazy `LazyGaussianMeanReverting`. Both produce the same process. The lazy version is more memory-efficient for long horizons. We should standardise on `LazyGaussianMeanReverting`.

In [ ]:
print("\n=== SUMMARY ===")
print("Low-level API verified: Market, FourHeap, Order, Agent all work")
print("MMEnv verified: reset/step/get_stats work")
print("SPEnv verified: reset/step work")
print("MMSPEnv verified: reset/step work")
print("\n6 API conflicts identified — see Part 3 and docs/decisions.md")
print("Next: design DefenderAgent + PredatorAgent + multi-agent wrapper")